# Using Precision as Optimizer

## Precision + Metaheuristic Algorithms

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import warnings

# ----------------------------
# Mealpy imports
# ----------------------------
from mealpy.swarm_based import PSO, BA, CSO, FA, ABC
from mealpy.evolutionary_based import GA
from mealpy.swarm_based.ACOR import OriginalACOR
from mealpy.utils.problem import FloatVar

warnings.filterwarnings('ignore')

# ----------------------------
# Load dataset
# ----------------------------
def load_data():
    amazon = pd.read_csv("amazon_cells_labelled.txt", sep="\t", header=None, names=["text", "label"])
    imdb   = pd.read_csv("imdb_labelled.txt", sep="\t", header=None, names=["text", "label"])
    yelp   = pd.read_csv("yelp_labelled.txt", sep="\t", header=None, names=["text", "label"])
    
    df = pd.concat([amazon, imdb, yelp], axis=0)
    from sklearn.feature_extraction.text import TfidfVectorizer
    vectorizer = TfidfVectorizer(stop_words="english", max_features=1000)
    X = vectorizer.fit_transform(df["text"]).toarray()
    y = df["label"].values
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    return X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = load_data()

# ----------------------------
# Classifier Objective Functions
# ----------------------------
def dt_objective(solution):
    max_depth = int(solution[0])
    min_samples_split = int(solution[1])
    min_samples_leaf = int(solution[2])
    criterion_idx = int(solution[3])
    criterion = 'gini' if criterion_idx == 0 else 'entropy'
    
    clf = DecisionTreeClassifier(
        max_depth=max_depth if max_depth > 0 else None,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        criterion=criterion,
        random_state=42
    )
    score = cross_val_score(clf, X_train, y_train, cv=3, scoring='precision')
    return -np.mean(score)  # minimize negative precision

def rf_objective(solution):
    n_estimators = int(solution[0])
    max_depth = int(solution[1])
    min_samples_split = int(solution[2])
    min_samples_leaf = int(solution[3])
    
    clf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth if max_depth > 0 else None,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=42
    )
    score = cross_val_score(clf, X_train, y_train, cv=3, scoring='precision')
    return -np.mean(score)

def gb_objective(solution):
    n_estimators = int(solution[0])
    learning_rate = solution[1]
    max_depth = int(solution[2])
    
    clf = GradientBoostingClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        random_state=42
    )
    score = cross_val_score(clf, X_train, y_train, cv=3, scoring='precision')
    return -np.mean(score)

def knn_objective(solution):
    n_neighbors = int(solution[0])
    p = int(solution[1])  # 1=Manhattan, 2=Euclidean
    
    clf = KNeighborsClassifier(
        n_neighbors=n_neighbors,
        p=p
    )
    score = cross_val_score(clf, X_train, y_train, cv=3, scoring='precision')
    return -np.mean(score)

def svm_objective(solution):
    C = solution[0]
    gamma = solution[1]
    
    clf = SVC(C=C, gamma=gamma, kernel='rbf', random_state=42)
    score = cross_val_score(clf, X_train, y_train, cv=3, scoring='precision')
    return -np.mean(score)

def xgb_objective(solution):
    n_estimators = int(solution[0])
    max_depth = int(solution[1])
    learning_rate = solution[2]
    
    clf = XGBClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42
    )
    score = cross_val_score(clf, X_train, y_train, cv=3, scoring='precision')
    return -np.mean(score)

# ----------------------------
# Problems Definition
# ----------------------------
problems = {
    "DecisionTree": {
        "obj_func": dt_objective,
        "bounds": [
            FloatVar(2, 30, "max_depth"),
            FloatVar(2, 10, "min_samples_split"),
            FloatVar(1, 5, "min_samples_leaf"),
            FloatVar(0, 1, "criterion")
        ]
    },
    "RandomForest": {
        "obj_func": rf_objective,
        "bounds": [
            FloatVar(10, 100, "n_estimators"),
            FloatVar(2, 30, "max_depth"),
            FloatVar(2, 10, "min_samples_split"),
            FloatVar(1, 5, "min_samples_leaf")
        ]
    },
    "GradientBoosting": {
        "obj_func": gb_objective,
        "bounds": [
            FloatVar(50, 200, "n_estimators"),
            FloatVar(0.01, 0.5, "learning_rate"),
            FloatVar(1, 10, "max_depth")
        ]
    },
    "KNN": {
        "obj_func": knn_objective,
        "bounds": [
            FloatVar(1, 20, "n_neighbors"),
            FloatVar(1, 2, "p")
        ]
    },
    "SVM": {
        "obj_func": svm_objective,
        "bounds": [
            FloatVar(0.1, 10, "C"),
            FloatVar(0.0001, 1, "gamma")
        ]
    },
    "XGBoost": {
        "obj_func": xgb_objective,
        "bounds": [
            FloatVar(50, 200, "n_estimators"),
            FloatVar(1, 10, "max_depth"),
            FloatVar(0.01, 0.5, "learning_rate")
        ]
    }
}

# ----------------------------
# Optimizers
# ----------------------------
optimizers = {
    "PSO": PSO.OriginalPSO(epoch=5, pop_size=10),
    "GA": GA.BaseGA(epoch=5, pop_size=10),
    "Bat": BA.OriginalBA(epoch=5, pop_size=10),
    "ACO": OriginalACOR(epoch=5, pop_size=10),
    "Cuckoo": CSO.OriginalCSO(epoch=5, pop_size=10),
    "Firefly": FA.OriginalFA(epoch=5, pop_size=10),
    "ABC": ABC.OriginalABC(epoch=5, pop_size=10)
}

# ----------------------------
# Run Optimization
# ----------------------------
final_results = {}

for clf_name, problem in problems.items():
    print(f"\n=== Optimizing {clf_name} ===")
    clf_results = {}
    
    for opt_name, optimizer in optimizers.items():
        print(f"Running {opt_name}...")
        best_agent = optimizer.solve(problem)
        best_solution = best_agent.solution
        best_precision = -best_agent.target.fitness
        clf_results[opt_name] = best_precision
        print(f"{opt_name} Best Precision: {best_precision:.4f}")
        print(f"{opt_name} Best Solution: {best_solution}\n")
    
    final_results[clf_name] = clf_results

# ----------------------------
# Visualization
# ----------------------------
for clf_name, clf_results in final_results.items():
    plt.figure(figsize=(8, 6))
    plt.bar(clf_results.keys(), clf_results.values())
    plt.title(f"{clf_name} Precision Comparison Across Metaheuristics")
    plt.ylabel("Precision")
    plt.xlabel("Optimizer")
    plt.show()
